# Cross-Sectional Equity Forecasting & Friction-Adjusted Long/Short Backtest
### Research Walkthrough & Pipeline Demonstration

---

## Executive Summary & Research Philosophy
In institutional quantitative finance, **the rigor of the research methodology is the true selling point**. A modest, honest Sharpe ratio (e.g. 0.5 – 1.0) derived from an un-leaked, friction-adjusted framework is infinitely more credible and defensible than a backtest boasting an annualized Sharpe of 3.0+ constructed with lookahead bias, unpenalized turnover, and survivorship-contaminated universes.

This notebook walks step-by-step through our end-to-end quantitative research framework:
1. **Universe Definition & Sector Mapping**: ~100 liquid US large/mid-cap equities across 11 GICS sectors (2012–2024), explicitly documenting survivorship bias.
2. **Data Ingestion & Parquet Caching**: Resilient OHLCV downloading with rate-limiting, retry logic, and trading-day gap analysis.
3. **Labeling via Market-Beta Stripping**: Estimating rolling 252-day OLS market beta vs. SPY to compute idiosyncratic residual returns $\varepsilon_{i,t}$, forecasting forward 5-day cumulative residual returns.
4. **Fractional Differentiation (FFD)**: Preserving long-memory while achieving stationarity through per-ticker ADF-validated optimal order $d$.
5. **Denoised Volatility & Liquidity Features**: Range-based volatility (Parkinson, Garman-Klass), Amihud illiquidity, and ATR regime ratios.
6. **Leakage-Safe Purged Cross-Validation**: Implementing `PurgedKFoldEmbargo` to prevent overlapping-label information contamination.
7. **Cross-Sectional Machine Learning**: LightGBM regressor with categorical sector encoding and purged hyperparameter tuning.
8. **Rank Information Coefficient (IC/IR)**: Evaluating cross-sectional rank correlation per rebalance period.
9. **Risk-Parity Long/Short Sizing**: Constructing dollar-neutral quintile portfolios sized by inverse trailing volatility ($w_i \propto 1/\sigma_i$).
10. **Vectorized Friction-Adjusted Backtest**: Simulating execution with realistic 8 bps costs, penalizing turnover, and generating 4-panel tearsheets.


## 1. Setup & Environment Configuration
We configure paths, verify Python version and packages, and configure plotting styles.


In [ ]:
import sys
import os
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"Python: {sys.version.split()[0]}")
print(f"Project root: {PROJECT_ROOT}")


## 2. Universe Construction & Survivorship Bias Caveat
A common flaw in academic and retail backtests is tech-heavy concentration (e.g. only trading AAPL, MSFT, TSLA) or selecting only today's S&P 500 winners without acknowledging survivorship bias.

Our universe covers **98 liquid large/mid-cap equities** across all 11 GICS sectors for 2012–2024, plus SPY and QQQ as benchmarks.


In [ ]:
from src.data.universe import UNIVERSE, BENCHMARKS, get_all_tickers, get_sector_map

all_tickers = get_all_tickers()
sector_map = get_sector_map()

sector_counts = pd.Series([sector_map[t] for t in all_tickers]).value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("Blues_r", len(sector_counts))
bars = ax.barh(sector_counts.index, sector_counts.values, color=colors, edgecolor='none')
ax.set_title("Universe Sector Distribution (98 Tickers Across 11 GICS Sectors)", fontsize=13, pad=12, fontweight='bold')
ax.set_xlabel("Number of Tickers")
for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f"{int(bar.get_width())}", 
            va='center', ha='left', fontsize=10, color='#333333')
ax.set_xlim(0, 24)
plt.tight_layout()
plt.show()

print(f"Total Tickers: {len(all_tickers)}")
print(f"Benchmarks: {BENCHMARKS}")


### Survivorship Bias Note
> **Research Caveat**: Because we select currently listed equities, tickers that went bankrupt, were acquired, or were delisted between 2012 and 2024 are excluded. In live production quantitative research, point-in-time historical constituent databases (such as Compustat Point-in-Time or survivorship-free CRSP data) are mandatory to eliminate this upward return bias.


## 3. Residual Return Labeling (Market Beta Stripping)
Predicting raw equity returns $r_{i,t}$ is problematic because stock movements are heavily dominated by the market common factor $r_{SPY,t}$. 
Forecasting models trained on raw returns primarily become crude market-timing models.

Instead, we target **idiosyncratic residual returns** by stripping rolling market beta:
$$r_{i,t} = \alpha_i + \beta_{i,t} \cdot r_{SPY,t} + \varepsilon_{i,t}$$
$$\beta_{i,t} = \frac{\text{Cov}_{252}(r_i, r_{SPY})}{\text{Var}_{252}(r_{SPY})}$$
The forward 5-day cumulative residual return is our target label:
$$y_{i,t} = \sum_{k=1}^{5} \varepsilon_{i, t+k}$$


In [ ]:
from src.features.labels import compute_rolling_beta, compute_residual_returns

# Synthetic example demonstrating beta stripping
np.random.seed(42)
dates = pd.date_range("2020-01-01", "2023-12-31", freq="B")
n = len(dates)

spy_ret = pd.Series(np.random.normal(0.0004, 0.01, n), index=dates)
stock_idio = pd.Series(np.random.normal(0.0001, 0.015, n), index=dates)
stock_ret = 1.35 * spy_ret + stock_idio  # true beta = 1.35

beta = compute_rolling_beta(stock_ret, spy_ret, window=252)
resids = compute_residual_returns(stock_ret, spy_ret, beta)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(dates, beta, color='#1f77b4', lw=1.5, label='Estimated Rolling 252d Beta')
ax1.axhline(1.35, color='red', linestyle='--', label='True Beta (1.35)')
ax1.set_title("Rolling Market Beta Estimation (SPY Factor Stripping)", fontweight='bold')
ax1.set_ylabel("Beta")
ax1.legend(loc='upper right')

ax2.plot(dates, stock_ret.cumsum(), color='gray', alpha=0.6, label='Raw Cumulative Return')
ax2.plot(dates, resids.cumsum(), color='#2ca02c', lw=1.5, label='Cumulative Residual Return (Idiosyncratic Alpha)')
ax2.set_title("Raw Return vs. Market-Stripped Idiosyncratic Residual", fontweight='bold')
ax2.set_ylabel("Cumulative")
ax2.legend(loc='upper left')
plt.tight_layout()
plt.show()


## 4. Fractional Differentiation (Preserving Memory & Stationarity)
Standard financial ML often differentiates prices via integer differencing ($d=1$, log returns). 
While $d=1$ guarantees stationarity, it **completely destroys long memory** (e.g. multi-month price patterns, support/resistance levels).
Conversely, raw log prices ($d=0$) preserve 100% of memory but are non-stationary, causing spurious ML correlations.

Using Marcos López de Prado's Fixed-Width Window Fractional Differentiation (FFD):
$$(1-B)^d = \sum_{k=0}^{\infty} (-1)^k \binom{d}{k} B^k = 1 - d B + \frac{d(d-1)}{2!} B^2 - \dots$$
We find the minimum $d \in [0.1, 0.9]$ that passes the Augmented Dickey-Fuller (ADF) test ($p < 0.05$).


In [ ]:
from src.features.fracdiff import get_weights_ffd, frac_diff_ffd_vectorized, find_optimal_d

# Visualize weight decay across differencing orders d
fig, ax = plt.subplots(figsize=(10, 4))
for d in [0.1, 0.3, 0.5, 0.7, 1.0]:
    w = get_weights_ffd(d, thresh=1e-4)
    ax.plot(range(min(len(w), 60)), w[:60], label=f"d = {d:.1f}", lw=1.8)

ax.set_title("FFD Memory Weight Decay by Differencing Order d", fontsize=12, fontweight='bold')
ax.set_xlabel("Lag k")
ax.set_ylabel("Weight w_k")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Denoised Volatility & Illiquidity Features
Rather than relying solely on simple close-to-close standard deviations, we compute:
1. **Parkinson Volatility**: Exploits daily High/Low prices to achieve an estimator with $\sim 5\times$ lower variance than close-to-close volatility:
$$\sigma_{\text{Parkinson}} = \sqrt{\frac{1}{4 \ln 2 \cdot n} \sum_{t=1}^n \left(\ln \frac{H_t}{L_t}\right)^2}$$
2. **Garman-Klass Volatility**: Incorporates Open, High, Low, and Close for even higher statistical efficiency.
3. **Amihud Illiquidity**: Measures price impact per unit of dollar volume:
$$\text{Illiq}_t = \frac{|r_t|}{\text{Volume}_t \times \text{Close}_t} \times 10^6$$
4. **Volatility Regime**: Ratio of 20-day ATR to 200-day ATR to identify regime shifts.


In [ ]:
from src.features.volatility import parkinson_volatility, garman_klass_volatility, amihud_illiquidity

# Generate sample price path
np.random.seed(101)
close_prices = 100 * np.exp(np.random.normal(0.0005, 0.015, 300).cumsum())
high_prices = close_prices * (1 + np.abs(np.random.normal(0, 0.01, 300)))
low_prices = close_prices * (1 - np.abs(np.random.normal(0, 0.01, 300)))
open_prices = low_prices + (high_prices - low_prices) * np.random.uniform(0.2, 0.8, 300)

p_vol = parkinson_volatility(high_prices, low_prices, window=20)
gk_vol = garman_klass_volatility(open_prices, high_prices, low_prices, close_prices, window=20)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(p_vol, label='Parkinson Volatility (High/Low, 20d)', color='#e377c2', lw=1.8)
ax.plot(gk_vol, label='Garman-Klass Volatility (OHLC, 20d)', color='#17becf', lw=1.8, linestyle='--')
ax.set_title("Range-Based Volatility Estimators", fontweight='bold')
ax.set_ylabel("Annualized Volatility")
ax.legend()
plt.tight_layout()
plt.show()


## 6. Leakage-Safe Validation: Purged K-Fold with Embargo
Standard cross-validation and naive `TimeSeriesSplit` suffer from severe information leakage when labels span multiple forward days ($h=5$).
If day $t$ is in the training set and day $t+1$ is in the test set, the label at day $t$ uses returns from $t+1 \dots t+5$, leaking test set price movements directly into training!

`PurgedKFoldEmbargo` eliminates this with:
1. **Purging**: Dropping all training samples in $[t_{\text{test,start}} - h, t_{\text{test,start}}]$.
2. **Embargo**: Dropping training samples in $[t_{\text{test,end}}, t_{\text{test,end}} + \text{embargo}]$ to eliminate autoregressive serial correlation.


In [ ]:
from src.validation.purged_cv import PurgedKFoldEmbargo

cv = PurgedKFoldEmbargo(n_splits=5, label_horizon=5, embargo_pct=0.01)
n_samples = 500
X = np.zeros((n_samples, 2))
dates = pd.date_range("2022-01-01", periods=n_samples, freq="B")

fig, ax = plt.subplots(figsize=(12, 4))
for fold, (train_idx, test_idx) in enumerate(cv.split(X, dates=dates)):
    # Plot train
    ax.scatter(train_idx, [fold] * len(train_idx), color='#1f77b4', s=10, marker='|')
    # Plot test
    ax.scatter(test_idx, [fold] * len(test_idx), color='#d62728', s=10, marker='|')

ax.set_yticks(range(5))
ax.set_yticklabels([f"Fold {i+1}" for i in range(5)])
ax.set_title("Purged K-Fold with Embargo Split Architecture (Blue=Train, Red=Test, Gaps=Purge/Embargo)", fontweight='bold')
ax.set_xlabel("Time Index")
plt.tight_layout()
plt.show()


## 7. Cross-Sectional Ranking & Information Coefficient (IC)
At each weekly rebalance date $t$:
1. We compute the cross-sectional Spearman rank correlation between predicted scores $\hat{y}_{i,t}$ and forward returns $y_{i,t}$:
$$\text{Rank IC}_t = \text{SpearmanCorr}(\hat{\mathbf{y}}_t, \mathbf{y}_t)$$
2. We aggregate across all $T$ rebalance periods:
$$\text{Mean IC} = \frac{1}{T}\sum_{t=1}^T \text{Rank IC}_t, \quad \text{IR} = \frac{\text{Mean IC}}{\text{Std}(\text{Rank IC})}, \quad t = \text{IR} \cdot \sqrt{T}$$

A mean Rank IC of **+0.03 to +0.06** with a $t$-statistic $> 2.0$ represents a strong, actionable institutional signal!


In [ ]:
from src.portfolio.ranking import compute_cross_sectional_ic, compute_ic_summary

# Simulate realistic cross-sectional predictions vs realizations across 20 periods x 80 stocks
np.random.seed(42)
records = []
rebal_dates = pd.date_range("2023-01-01", periods=50, freq="W-FRI")
tickers = [f"TICK_{i:02d}" for i in range(80)]

for d in rebal_dates:
    # True signal with noise: IC ~ 0.04
    true_alpha = np.random.normal(0, 0.02, len(tickers))
    noise = np.random.normal(0, 0.08, len(tickers))
    pred = true_alpha + noise
    realized = true_alpha + np.random.normal(0, 0.08, len(tickers))
    for t, p, r in zip(tickers, pred, realized):
        records.append({"date": d, "ticker": t, "predicted_score": p, "fwd_raw_return": r})

demo_df = pd.DataFrame(records)
ic_series = compute_cross_sectional_ic(demo_df, "predicted_score", "fwd_raw_return")
summary = compute_ic_summary(ic_series)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(ic_series["date"], ic_series["ic"], width=5, 
       color=np.where(ic_series["ic"] >= 0, '#2ca02c', '#d62728'), alpha=0.8)
ax.axhline(summary["mean_ic"], color='navy', linestyle='--', 
           label=f"Mean IC: {summary['mean_ic']:+.4f} (IR: {summary['ir']:+.2f}, t-stat: {summary['ic_tstat']:+.2f})")
ax.set_title("Simulated Weekly Cross-Sectional Rank IC Time Series", fontweight='bold')
ax.set_ylabel("Rank IC")
ax.legend()
plt.tight_layout()
plt.show()


## 8. Dollar-Neutral Portfolio Sizing (Volatility-Parity)
To build a market-neutral strategy:
1. **Quintile Selection**: Long the top 20% (Q5) highest-ranked stocks, short the bottom 20% (Q1).
2. **Inverse-Volatility Weighting**: Within each leg, weight stocks inversely proportional to their 20-day trailing volatility:
$$w_i \propto \frac{1}{\sigma_i}$$
3. **Leg Normalization**:
$$\sum_{i \in \text{Long}} w_i = +1.0, \quad \sum_{j \in \text{Short}} w_j = -1.0$$
$$\text{Net Exposure} = 0.0, \quad \text{Gross Exposure} = 2.0$$


In [ ]:
from src.portfolio.sizing import construct_portfolio, verify_portfolio_weights

# Demo portfolio sizing on one cross-section
cross_section = pd.DataFrame({
    "date": pd.Timestamp("2023-06-01"),
    "ticker": [f"STOCK_{i}" for i in range(25)],
    "predicted_score": np.random.normal(0, 1, 25),
    "realized_vol_20": np.random.uniform(0.15, 0.45, 25),
})

pf = construct_portfolio(cross_section, score_col="predicted_score", vol_col="realized_vol_20", rebalance_freq=1)
longs = pf[pf["weight"] > 0].sort_values("weight", ascending=False)
shorts = pf[pf["weight"] < 0].sort_values("weight", ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(longs["ticker"], longs["weight"] * 100, color='#2ca02c')
ax1.set_title("Long Leg Weights (Sum = +100%)", fontweight='bold')
ax1.set_ylabel("Weight (%)")
ax1.tick_params(axis='x', rotation=45)

ax2.bar(shorts["ticker"], shorts["weight"] * 100, color='#d62728')
ax2.set_title("Short Leg Weights (Sum = -100%)", fontweight='bold')
ax2.set_ylabel("Weight (%)")
ax2.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


## 9. Vectorized Friction-Adjusted Backtest Engine
A quantitative model that is profitable gross of fees can easily lose money net of costs.
Our engine penalizes every rebalance for turnover:
$$\text{Turnover}_t = \sum_i |w_{i,t} - w_{i,t-1}|$$
$$\text{Cost}_t = \text{Turnover}_t \times \frac{\text{Total Cost Bps}}{10{,}000}$$
$$\text{Net Return}_t = \text{Gross Return}_t - \text{Cost}_t$$

We benchmark our dollar-neutral strategy against **SPY** (S&P 500) and **QQQ** (Nasdaq-100).


In [ ]:
from src.backtest.engine import compute_portfolio_returns, compute_performance_metrics

# Demonstrate return and metrics calculation on synthetic returns
np.random.seed(42)
n_days = 500
test_dates = pd.date_range("2023-01-01", periods=n_days, freq="B")
gross_rets = pd.Series(np.random.normal(0.0006, 0.008, n_days), index=test_dates)
# Rebalance every 5 days with turnover ~ 0.4
turnover = pd.Series(0.0, index=test_dates)
turnover.iloc[::5] = 0.40
costs = turnover * (8.0 / 10000)
net_rets = gross_rets - costs

metrics_gross = compute_performance_metrics(gross_rets)
metrics_net = compute_performance_metrics(net_rets)

summary_table = pd.DataFrame({
    "Gross of Fees": metrics_gross,
    "Net of Fees (8 bps)": metrics_net
})
print("Performance Comparison:")
print(summary_table.to_string())


## 10. Running the Full Pipeline
To execute the entire automated pipeline end-to-end (downloading cached universe data, stripping beta, fitting LightGBM with Purged CV, and producing the 4-panel tearsheet):

```bash
python -m src.run_pipeline
```

Outputs will be generated in `outputs/`:
- `outputs/tearsheet.png`: 4-panel visual performance report
- `outputs/daily_returns.csv`: Time series of gross/net returns and turnover
- `outputs/results_summary.json`: Detailed IC metrics, hyperparameter configs, and Sharpe ratios
- `models/lightgbm_model.pkl`: Serialized model checkpoint
- `models/feature_importances.csv`: Feature ranking table


## 11. Conclusion & Next Steps
This framework demonstrates institutional quantitative rigor across all key dimensions:
- **No Lookahead Bias**: Explicit temporal split and market beta stripping.
- **No Overlapping Leakage**: Validated Purged K-Fold with Embargo.
- **Stationarity & Memory**: Fixed-Width Window Fractional Differentiation.
- **Realistic Friction**: Turnover tracking and 8 bps transaction cost drag.
- **Risk Control**: Volatility-parity dollar-neutral portfolio construction.

### Production Enhancements:
1. **Point-in-Time Universe**: Integrate survivorship-free index constituent history.
2. **Multi-Factor Risk Model**: Extend beyond single-factor CAPM beta to Barra/Fama-French 5-factor beta stripping.
3. **Execution Optimization**: Quadratic programming (Markowitz with transaction cost penalty) instead of simple quintile sorting.
